In [10]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd

## Load trained model

In [ ]:
BEST_MODEL_PATH = Path.cwd().parent / "runs" / "detect" / "train" / "weights" / "best.pt"
model = YOLO(BEST_MODEL_PATH)

In [5]:
DATASET_ROOT = Path.cwd().parent / "data" / "ElectroCom61 A Multiclass Dataset for Detection of Electronic Components" / "ElectroCom-61_v2" / "data.yaml"

## Run validation

In [6]:
metrics = model.val(data=DATASET_ROOT)

Ultralytics 8.4.128  Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 8151MiB)
YOLO11s summary (fused): 101 layers, 9,436,407 parameters, 0 gradients, 21.5 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 722.3218.3 MB/s, size: 63.8 KB)
val: Scanning C:\Users\adely\Desktop\blablabla\electronic-component-classifier\data\ElectroCom61 A Multiclass Dataset for Detection of Electronic Components\ElectroCom-61_v2\valid\labels.cache... 649 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 649/649 226.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 6.1it/s 6.7s0.2s
                   all        649       3557      0.885      0.895      0.922      0.612
      1-5-Volt-Battery        103        109      0.963      0.982      0.984      0.647
      3-3-Volt-Battery         38         38      0.929      0.947      0.929       0.65
     7-Segment-Display         36         36      0.936      0

In [26]:
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

Precision: 0.885
Recall: 0.895
mAP50: 0.922
mAP50-95: 0.612


## Viewing the metrics for each class

In [28]:
rows = []

for i, class_id in enumerate(metrics.box.ap_class_index):
    rows.append({
        "class": model.names[class_id],
        "precision": metrics.box.p[i],
        "recall": metrics.box.r[i],
        "mAP50": metrics.box.ap50[i],
        "mAP50-95": metrics.box.ap[i]
    })

metrics_df = pd.DataFrame(rows)
sorted_metrics_df = metrics_df.sort_values("mAP50")
sorted_metrics_df.head(15)

,class,precision,recall,mAP50,mAP50-95
60,Zener-Diode,0.582683,0.727273,0.649082,0.306057
56,Tact-Switch,0.617395,0.530435,0.653507,0.306947
7,BJT-Transistor,0.723335,0.610000,0.709459,0.384833
38,Low-Voltage-Ceramic-Capacitor,0.685447,0.568627,0.711131,0.346817
12,Buzzer,0.642417,0.617647,0.713243,0.375107
34,Keypad,0.723461,0.791209,0.728044,0.473041
31,IGBT,0.592886,0.832182,0.761215,0.408197
40,MOSFET,0.843328,0.579703,0.786911,0.434295
46,Push-Switch,0.780148,0.692552,0.827404,0.463824
39,MLC-Capacitor,0.781601,0.954551,0.860897,0.473621


## Evaluation

**Overall**: precision 0.885, recall 0.895, mAP50 0.922, mAP50-95 0.612 on the validation set (yolo11s, 640px, trained on the cleaned dataset)

**Overfitting check**: train losses decreased smoothly throughout training, but val losses bottomed out around epoch 10 and increased for the rest of the run indicating mild overfitting on the loss metrics. However the precision, recall, mAP50 and mAP50-95 metrics for val remained stable so the best metric (mAP50-95) wasn't affected in practice.

**Worst performing classes**: Zener-Diode, Tact-Switch, BJT-Transistor, Low-Voltage-Ceramic-Capacitor, Buzzer, Keypad, IGBT, MOSFET, Push-Switch, MLC-Capacitor, Fuse, LDR-Sensor, Capacitor-10mf, Diode, TCRT5000.

**Cross-checked against the EDA bounding box size prediction** (smallest median box area per class): 9 out of the 15 classes predicted to be at risk actually underperformed compared to other classes (Tact-Switch, BJT-Transistor, Low-Voltage-Ceramic-Capacitor, Buzzer, Push-Switch, MLC-Capacitor, Fuse, LDR-Sensor, TCRT5000). The other 6 predicted classes performed well despite their small sizes, the extra model capacity of yolo11s compensating for their small size compared to smaller models such as yolo11n.

**Second, unpredicted failure**: Zener-Diode, IGBT, MOSFET, Capacitor-10mf, Diode all underperform despite not being flagged by their box size. These components are visually similar and the confusion matrix shows an off-diagonal signal between IGBT and MOSFET, supporting class similarity confusion rather than a small object problem.

**Third, unexplained failure**: Another class with low performance was Keypad, the confusion matrix showing that it got confused for background instead.

**Conclusion**: there are three distinct failure modes that account for most of the underperforming classes, these being small objects (which was partially mitigated by using a larger model), visually similar class components (not mitigated by model size, would need targeted data or features) and the Keypad class components getting confused for background rather than another class or explained by box size, and remains unexplained within this project's scope. Most of these failures are understood, documented limitations rather than unexplained gaps, with the exception of Keypad, which remains unresolved.